In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm.notebook import tqdm

print(f"PyTorch version: {torch.__version__}")

In [ ]:
## Configuration
OUTPUT_DIR      = './roberta_model_outputs'
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'best_roberta_model.pt')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME  = 'roberta-base'
MAX_LENGTH  = 256
NUM_CLASSES = 3

BATCH_SIZE    = 16
LEARNING_RATE = 1e-5
NUM_EPOCHS    = 4
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 0.01
DROPOUT_RATE  = 0.4
EARLY_STOPPING_PATIENCE = 2
GRADIENT_CLIP = 1.0

RANDOM_STATE = 42
LABEL_MAP   = {'liberal': 0, 'conservative': 1, 'center': 2}
LABEL_NAMES = ['Liberal', 'Conservative', 'Center']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {DEVICE}")

In [ ]:
class NewsDataset(Dataset):
    """Custom Dataset for RoBERTa input conversion"""
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens   = True,
            max_length           = self.max_length,
            padding              = 'max_length',
            truncation           = True,
            return_attention_mask= True,
            return_tensors       = 'pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
def load_and_preprocess_data(data_path):
    df = pd.read_csv(data_path)
    required = ['Title', 'News_Body', 'Label']
    df = df.dropna(subset=required)
    df['Label'] = df['Label'].str.strip().str.lower()
    
    for col in ['Stance', 'issue', 'topic', 'roundup']:
        if col not in df.columns: df[col] = ''
        df[col] = df[col].fillna('')

    def build_train_text(row):
        return str(row['Title']) + ' </s> ' + str(row['News_Body'])

        return ' </s> '.join(parts)

    df['train_text'] = df.apply(build_train_text, axis=1)
    df['label_encoded'] = df['Label'].map(LABEL_MAP)
    df = df.dropna(subset=['label_encoded'])
    df['label_encoded'] = df['label_encoded'].astype(int)

    return df['train_text'].values, df['label_encoded'].values, df

In [ ]:
DATA_PATH = 'cleaned_political_bias_data (1).csv'
texts, labels, df_full = load_and_preprocess_data(DATA_PATH)

# Splits
X_tv, X_test, y_tv, y_test = train_test_split(texts, labels, test_size=0.15, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=42, stratify=y_tv)

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(NewsDataset(X_train, y_train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NewsDataset(X_val,   y_val,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE)
test_loader  = DataLoader(NewsDataset(X_test,  y_test,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE)

print(f"Prepared {len(X_train)} train, {len(X_val)} val, {len(X_test)} test samples.")

In [ ]:
import torch.nn as nn
class RobertaNewsClassifier(nn.Module):
    """Fine-tuning RoBERTa for 3-class classification"""
    def __init__(self, num_classes=3, dropout=0.3):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # RoBERTa uses the summary of the <s> token
        pooled  = outputs.pooler_output
        pooled  = self.dropout(pooled)
        return self.classifier(pooled)

model = RobertaNewsClassifier().to(DEVICE)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    losses, correct, total = [], 0, 0
    for batch in tqdm(loader, desc="Training"):
        ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        losses.append(loss.item())
        correct += (logits.argmax(1) == lbls).sum().item()
        total += lbls.size(0)
    return np.mean(losses), correct / total

def eval_model(model, loader, criterion, device):
    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for batch in loader:
            ids, mask, lbls = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
            logits = model(ids, mask)
            loss = criterion(logits, lbls)
            losses.append(loss.item())
            correct += (logits.argmax(1) == lbls).sum().item()
            total += lbls.size(0)
    return np.mean(losses), correct / total

In [ ]:
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

best_acc = 0
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_loss, val_acc = eval_model(model, val_loader, criterion, DEVICE)
    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("Saved new best model!")

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in test_loader:
        ids, mask, lbls = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['label'].to(DEVICE)
        logits = model(ids, mask)
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(lbls.cpu().numpy())

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))